# 语雀 (Yuque) Web API 全流程验证

> **目标**：验证语雀 Web API 的完整能力链路(Cookie 认证,免费方案)
> **覆盖**：认证 → 用户信息 → 知识库 → 创建 → 更新(文本/标题/列表/代码/公式/表格/图片) → 读取 → 搜索 → 删除
> **认证**：`_yuque_session` Cookie + `ctoken`(无需会员 Token)

> **注意**：本 Notebook 使用的是语雀 Web 端内部 API,通过浏览器 Cookie 认证. 这些 API 非官方开放接口,可能随语雀版本更新而变化. 

In [1]:
import os
import json
import time
import requests
from datetime import datetime
from pathlib import Path

class YuqueClient:
    """语雀 Web API 客户端(Cookie 认证)"""
    BASE_URL = "https://www.yuque.com"

    def __init__(self, session=None, ctoken=None):
        self.session = session or os.environ.get("YUQUE_SESSION")
        self.ctoken = ctoken or os.environ.get("YUQUE_CTOKEN")
        if not self.session:
            raise ValueError("请提供 _yuque_session Cookie 或设置 YUQUE_SESSION 环境变量")
        self.cookies = {
            "_yuque_session": self.session,
            "ctoken": self.ctoken or ""
        }
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Referer": "https://www.yuque.com",
            "Content-Type": "application/json"
        }
        self.session_req = requests.Session()
        self.session_req.cookies.update(self.cookies)
        self.session_req.headers.update(self.headers)
        self._doc_cache = {}  # 缓存 doc_id -> {slug, book_id}

    def request(self, method, endpoint, **kwargs):
        """发送 HTTP 请求"""
        url = f"{self.BASE_URL}{endpoint}"
        resp = self.session_req.request(method, url, **kwargs)
        resp.raise_for_status()
        return resp.json()

    def health_check(self):
        """健康检查：验证 Cookie 是否有效"""
        try:
            result = self.request("GET", "/api/mine")
            user = result.get("data", {})
            return {
                "ok": True,
                "login": user.get("login"),
                "name": user.get("name"),
                "user_id": user.get("id")
            }
        except Exception as e:
            return {"ok": False, "error": str(e)}

    def get_user(self):
        return self.request("GET", "/api/mine")

    def get_books(self):
        return self.request("GET", "/api/mine/books")

    def get_docs(self, book_id):
        return self.request("GET", f"/api/docs?book_id={book_id}")

    def get_doc(self, slug, book_id):
        return self.request("GET", f"/api/docs/{slug}?book_id={book_id}")

    def create_doc(self, book_id, title, body="", target_uuid="", doc_type="Doc"):
        """创建文档"""
        payload = {
            "action": "prependChild" if not target_uuid else "prependChild",
            "book_id": book_id,
            "title": title,
            "type": doc_type,
            "insert_to_catalog": True,
            "slug": "",
            "status": 0,
            "body_draft_asl": None
        }
        if target_uuid:
            payload["target_uuid"] = target_uuid
        return self.request("POST", "/api/docs", json=payload)

    def update_doc(self, doc_id, title=None, body=None):
        """更新文档(支持 markdown body)"""
        payload = {}
        if title is not None:
            payload["title"] = title
        if body is not None:
            payload["body"] = body
        return self.request("PUT", f"/api/docs/{doc_id}", json=payload)

    def append_doc_body(self, doc_id, new_content):
        """读取现有文档内容,追加新 markdown 内容"""
        doc = self.get_doc_by_id(doc_id)
        current_body = doc.get("body", "") or ""
        updated_body = current_body + "\n\n" + new_content if current_body else new_content
        return self.update_doc(doc_id, body=updated_body)

    def get_doc_by_id(self, doc_id):
        """通过 ID 获取文档(需要先获取 slug 再查询)"""
        if hasattr(self, "_doc_cache") and doc_id in self._doc_cache:
            slug = self._doc_cache[doc_id]["slug"]
            book_id = self._doc_cache[doc_id]["book_id"]
            return self.get_doc(slug, book_id).get("data", {})
        return {}

    def delete_doc(self, doc_id):
        return self.request("DELETE", f"/api/docs/{doc_id}")

    def search(self, query, search_type="content", tab="public", limit=10, page=1):
        """搜索"""
        params = {
            "q": query,
            "type": search_type,  # content, book, user
            "tab": tab,
            "limit": limit,
            "p": page
        }
        return self.request("GET", "/api/zsearch", params=params)

    def lock_doc(self, doc_id, uuid_str):
        return self.request("PUT", f"/api/docs/{doc_id}/lock", json={"uuid": uuid_str})

# 初始化客户端(自动从环境变量读取)
client = YuqueClient()

# 健康检查
health = client.health_check()
print(f"[{'OK' if health['ok'] else 'FAIL'}] YuqueClient initialized")
if health['ok']:
    print(f"       Login: {health['login']}")
    print(f"       Name: {health['name']}")
    print(f"       User ID: {health['user_id']}")
else:
    print(f"       Error: {health.get('error', 'Unknown')}")

d:\python-envs\baseenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


[OK] YuqueClient initialized
       Login: lebornyuan
       Name: LeBornYuan
       User ID: 42982692


## 1. 获取用户信息

验证 `GET /api/mine` 获取当前登录用户信息. 

In [2]:
user_result = client.get_user()
user = user_result['data']

MY_LOGIN = user['login']
MY_USER_ID = user['id']
MY_NAME = user['name']

print(f"[OK] User info fetched")
print(f"     Login: {MY_LOGIN}")
print(f"     Name: {MY_NAME}")
print(f"     User ID: {MY_USER_ID}")
print(f"     Docs count: {user.get('statistics', {}).get('doc_count', 'N/A')}")

[OK] User info fetched
     Login: lebornyuan
     Name: LeBornYuan
     User ID: 42982692
     Docs count: 1753


## 2. 获取知识库列表

验证 `GET /api/mine/books` 获取当前用户知识库列表. 

> 语雀结构：**用户 → 知识库(Book) → 文档(Doc)**

In [3]:
books_result = client.get_books()
books = books_result.get('data', [])

print(f"[OK] Found {len(books)} books")
for i, book in enumerate(books[:5]):
    print(f"  {i+1}. {book['name']} (slug: {book['slug']}, id: {book['id']}, docs: {book['items_count']})")

# 选择第一个自己的知识库作为测试目标
my_books = [b for b in books if b.get('user_id') == MY_USER_ID]
if not my_books:
    raise Exception('未找到属于自己的知识库')

TEST_BOOK = my_books[0]
TEST_BOOK_ID = TEST_BOOK['id']
TEST_BOOK_SLUG = TEST_BOOK['slug']
TEST_NAMESPACE = f"{MY_LOGIN}/{TEST_BOOK_SLUG}"

print(f"\n[OK] Selected test book: {TEST_BOOK['name']}")
print(f"     Book ID: {TEST_BOOK_ID}")
print(f"     Namespace: {TEST_NAMESPACE}")

[OK] Found 6 books
  1. Awesome-CS336-NoteForEveryone (slug: zf1hbk, id: 68016047, docs: 38)
  2. LLM从入门到入土 (slug: qah8x7, id: 68025057, docs: 234)
  3. LLM知识全景 (slug: tp8id6, id: 67989174, docs: 678)
  4. 0802 (slug: ucamg9, id: 67989147, docs: 803)
  5. Java 8 Guwen K (slug: ydpq6c, id: 50606089, docs: 1351)

[OK] Selected test book: Awesome-CS336-NoteForEveryone
     Book ID: 68016047
     Namespace: lebornyuan/zf1hbk


## 3. 创建文档

验证 `POST /api/docs` 创建新文档. 

> ⚠️ 注意：语雀免费账户有文档数量限制(通常单知识库 100 篇),如果创建失败请检查是否已超限. 

In [4]:
title = f"API验证 - {datetime.now().strftime('%H:%M:%S')}"

try:
    create_result = client.create_doc(TEST_BOOK_ID, title=title)
    doc_data = create_result['data']
    DOC_ID = doc_data['id']
    DOC_SLUG = doc_data['slug']
    DOC_TITLE = doc_data['title']
    
    # 缓存文档信息供后续使用
    client._doc_cache = {DOC_ID: {'slug': DOC_SLUG, 'book_id': TEST_BOOK_ID}}
    
    print(f"[OK] Created doc: {DOC_TITLE}")
    print(f"     doc_id: {DOC_ID}")
    print(f"     slug: {DOC_SLUG}")
    print(f"     URL: https://yuque.com/{TEST_NAMESPACE}/{DOC_SLUG}")
except Exception as e:
    print(f"[WARN] Create doc failed: {e}")
    print("       可能原因：文档数量超限,将使用现有文档进行后续测试")
    # 获取现有文档列表,选择第一个进行测试
    docs_result = client.get_docs(TEST_BOOK_ID)
    docs = docs_result.get('data', [])
    if not docs:
        raise Exception('知识库中没有文档,无法继续测试')
    existing_doc = docs[0]
    DOC_ID = existing_doc['id']
    DOC_SLUG = existing_doc['slug']
    DOC_TITLE = existing_doc['title']
    client._doc_cache = {DOC_ID: {'slug': DOC_SLUG, 'book_id': TEST_BOOK_ID}}
    print(f"[OK] Using existing doc: {DOC_TITLE} (id: {DOC_ID})")

[OK] Created doc: API验证 - 22:59:15
     doc_id: 268176266
     slug: gc03wrf464a5gduh
     URL: https://yuque.com/lebornyuan/zf1hbk/gc03wrf464a5gduh


## 4. 追加文本内容

语雀 Web API 通过 `PUT /api/docs/{id}` 的 `body` 字段更新文档内容,
直接传入 **Markdown** 格式字符串即可. 

追加逻辑：先 GET 现有 body → 拼接新内容 → PUT 更新. 

In [5]:
text_content = """
## 文本追加测试

这是一段普通文本,用于验证语雀 API 的 markdown 内容更新. 

- 支持无序列表
- 自动渲染为语雀列表样式

1. 有序列表项1
2. 有序列表项2

> 引用块测试

段落之间需要空行分隔. 
"""

result = client.append_doc_body(DOC_ID, text_content.strip())
print(f"[OK] Appended text content to doc {DOC_ID}")

[OK] Appended text content to doc 268176266


## 5. 追加代码块

Markdown 代码块语法(```language)会被语雀自动渲染为代码块. 

In [6]:
code_content = """
## 代码块测试

Python 示例代码：

```python
def hello():
    print('Hello, Yuque!')

hello()
```
"""

client.append_doc_body(DOC_ID, code_content.strip())
print("[OK] Appended code block")

[OK] Appended code block


## 6. 追加数学公式

语雀原生支持 **LaTeX** 公式语法：
- 行内公式：`$...$`
- 块级公式：`$$...$$`

> **注意**：公式内容使用标准 LaTeX 语法,`\frac` 等命令使用单反斜杠即可. 

In [7]:
formula_content = r"""
## 数学公式测试

一元二次方程求根公式：

$$x = \frac{-b \pm \sqrt{b^2-4ac}}{2a}$$
"""

client.append_doc_body(DOC_ID, formula_content.strip())
print("[OK] Appended formula block")

[OK] Appended formula block


### 6.1 GRPO Reward 与 KL 散度

GRPO(Group Relative Policy Optimization)的核心目标函数. 

In [8]:
grpo_content = r"""
### GRPO 目标函数

策略比率与裁剪：

$$r_i(\theta) = \frac{\pi_{\theta}(o_i|q)}{\pi_{\theta_{old}}(o_i|q)}$$

组内相对优势(组大小 G)：

$$\hat{A}_i = \frac{r_i - \text{mean}(\{r_j\}_{j=1}^{G})}{\text{std}(\{r_j\}_{j=1}^{G})}$$

带 KL 惩罚的 GRPO 目标：

$$J_{GRPO}(\theta) = \mathbb{E}_{q,\{o_i\}}\left[ \frac{1}{G}\sum_{i=1}^{G}\left( \min\left( r_i\hat{A}_i, \text{clip}(r_i, 1-\epsilon, 1+\epsilon)\hat{A}_i \right) - \beta D_{KL}(\pi_{\theta} \| \pi_{ref}) \right) \right]$$
"""

client.append_doc_body(DOC_ID, grpo_content.strip())
print("[OK] Appended GRPO formulas")

[OK] Appended GRPO formulas


### 6.2 偏微分方程(热方程)

In [9]:
pde_content = r"""
### 热传导方程

三维热方程：

$$\frac{\partial u}{\partial t} = \alpha \nabla^2 u = \alpha \left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} + \frac{\partial^2 u}{\partial z^2} \right)$$

初始条件与边界条件：

$$u(x,y,z,0) = \varphi(x,y,z), \quad u|_{\partial \Omega} = g(x,y,z,t)$$

分离变量法得到的解析解：

$$u(\mathbf{x},t) = \sum_{n=1}^{\infty} A_n e^{-\alpha \lambda_n t} \phi_n(\mathbf{x})$$
"""

client.append_doc_body(DOC_ID, pde_content.strip())
print("[OK] Appended PDE formulas")

[OK] Appended PDE formulas


### 6.3 数论公式

In [10]:
nt_content = r"""
### 数论核心公式

黎曼 ζ 函数的函数方程：

$$\zeta(s) = 2^{s} \pi^{s-1} \sin\left( \frac{\pi s}{2} \right) \Gamma(1-s) \zeta(1-s)$$

质数定理(素数计数函数)：

$$\pi(x) \sim \frac{x}{\ln x} \sim \text{Li}(x) = \int_{2}^{x} \frac{dt}{\ln t}$$

欧拉乘积公式：

$$\zeta(s) = \sum_{n=1}^{\infty} \frac{1}{n^{s}} = \prod_{p \text{ prime}} \frac{1}{1 - p^{-s}}, \quad \text{Re}(s) > 1$$
"""

client.append_doc_body(DOC_ID, nt_content.strip())
print("[OK] Appended number theory formulas")

[OK] Appended number theory formulas


### 6.4 Scaled Dot-Product Attention

In [11]:
attn_content = r"""
### Scaled Dot-Product Attention

核心公式：

$$\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{QK^{T}}{\sqrt{d_k}} \right) V$$

Mask 版本(因果/自回归)：

$$\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{QK^{T}}{\sqrt{d_k}} + M \right) V, \quad M_{ij} = \begin{cases} 0 & i \geq j \\ -\infty & i < j \end{cases}$$
"""

client.append_doc_body(DOC_ID, attn_content.strip())
print("[OK] Appended Attention formula")

[OK] Appended Attention formula


### 6.5 Multi-Head Attention

In [12]:
mha_content = r"""
### Multi-Head Attention

第 i 个注意力头的线性投影：

$$Q_i = X W_i^{Q}, \quad K_i = X W_i^{K}, \quad V_i = X W_i^{V}$$

单头注意力输出：

$$\text{head}_i = \text{Attention}(Q_i, K_i, V_i) = \text{softmax}\left( \frac{Q_i K_i^{T}}{\sqrt{d_k}} \right) V_i$$

多头拼接与输出投影：

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^{O}$$

维度关系(d_model = h × d_k)：

$$W_i^{Q}, W_i^{K}, W_i^{V} \in \mathbb{R}^{d_{model} \times d_k}, \quad W^{O} \in \mathbb{R}^{hd_k \times d_{model}}$$
"""

client.append_doc_body(DOC_ID, mha_content.strip())
print("[OK] Appended MHA formulas")

[OK] Appended MHA formulas


### 6.6 MoE Auxiliary Loss

In [13]:
moe_content = r"""
### MoE Auxiliary Loss

专家 i 的 token 比例(实际路由到的比例)：

$$f_i = \frac{1}{T} \sum_{x \in \mathcal{B}} \mathbb{1}\{ \text{argmax}_j \, p_j(x) = i \}$$

专家 i 的平均路由概率(路由器分配的期望)：

$$P_i = \frac{1}{T} \sum_{x \in \mathcal{B}} p_i(x)$$

Auxiliary Loss(负载均衡惩罚)：

$$\mathcal{L}_{aux} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

总损失(主损失 + 辅助损失)：

$$\mathcal{L}_{total} = \mathcal{L}_{main} + \mathcal{L}_{aux}$$
"""

client.append_doc_body(DOC_ID, moe_content.strip())
print("[OK] Appended MoE AuxLoss formulas")

[OK] Appended MoE AuxLoss formulas


## 7. 追加表格

Markdown 表格语法会被语雀自动渲染为表格. 

In [14]:
table_content = """
## 表格测试

| 姓名 | 年龄 | 城市 |
|------|------|------|
| Alice | 25 | 北京 |
| Bob | 30 | 上海 |
| Carol | 28 | 深圳 |
"""

client.append_doc_body(DOC_ID, table_content.strip())
print("[OK] Appended table")

[OK] Appended table


## 8. 准备测试图片

使用 Pillow 生成本地测试图片(和飞书测试保持一致). 

In [15]:
from PIL import Image, ImageDraw, ImageFont
import io

img = Image.new('RGB', (400, 200), color=(73, 109, 137))
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype("arial.ttf", 24)
except:
    font = ImageFont.load_default()
draw.text((20, 80), "Test Image for Yuque", fill=(255, 255, 255), font=font)

buf = io.BytesIO()
img.save(buf, format='PNG')
img_bytes = buf.getvalue()

print(f"[OK] Test image created: {len(img_bytes)} bytes")

[OK] Test image created: 4536 bytes


## 9. 插入网络图片

语雀 Markdown 原生支持 `![alt](url)` 语法引用外部图片. 

> **关于本地图片上传**：语雀 Web API 未提供公开的本地图片上传接口. 如果需要将本地图片上传到语雀 CDN,建议：
> 1. 使用语雀 Web 编辑器手动粘贴/上传图片
> 2. 或先将图片上传到外部图床,再引用 URL 到文档中
> 3. 或使用 `POST /api/import?ctoken=...` 导入包含图片的 markdown 文件

In [16]:
import requests as req

IMAGE_URL = "https://picsum.photos/800/400"

# 下载网络图片验证可访问性
print(f"--- 正在验证图片 URL: {IMAGE_URL} ---")
img_resp = req.get(IMAGE_URL, timeout=30)
img_resp.raise_for_status()
print(f"✅ 图片可访问: {len(img_resp.content)} bytes")

# 将网络图片插入文档(markdown 图片语法)
image_md = f"""
## 图片测试

### 网络图片

![随机风景图]({IMAGE_URL})

> 图1: 通过网络 URL 插入的图片

### 本地图片说明

本地生成的测试图片({len(img_bytes)} bytes)已准备就绪. 
由于语雀 Web API 未暴露公开的图片上传端点,本地图片需要通过以下方式处理：
- 上传到外部图床后引用 URL
- 在语雀 Web 编辑器中手动粘贴上传
"""

client.append_doc_body(DOC_ID, image_md.strip())
print("[OK] Appended image markdown")

--- 正在验证图片 URL: https://picsum.photos/800/400 ---
✅ 图片可访问: 54684 bytes
[OK] Appended image markdown


## 10. 读取文档

验证 `GET /api/docs/{slug}?book_id={book_id}` 读取文档详情和内容. 

In [17]:
read_result = client.get_doc(DOC_SLUG, TEST_BOOK_ID)
doc_data = read_result.get('data', {})

print(f"[OK] Read doc: {doc_data.get('title')}")
print(f"     Format: {doc_data.get('format')}")
print(f"     Word count: {doc_data.get('word_count')}")
print(f"     Content length: {len(doc_data.get('body', '') or doc_data.get('content', ''))} chars")

# 打印 body 前 500 字符
body_preview = doc_data.get('body', '') or doc_data.get('content', '')
if body_preview:
    print('\n--- Body Preview (first 500 chars) ---')
    print(body_preview[:500])

[OK] Read doc: API验证 - 22:59:15
     Format: lake
     Word count: 102
     Content length: 1410 chars

--- Body Preview (first 500 chars) ---
<!doctype lake><meta name="doc-version" content="1" /><meta name="viewport" content="fixed" /><meta name="typography" content="classic" /><meta name="paragraphSpacing" content="relax" /><h2 data-lake-id="pbDdU" id="pbDdU"><span data-lake-id="uf41aefaf" id="uf41aefaf">图片测试</span></h2><h3 data-lake-id="BHCVH" id="BHCVH"><span data-lake-id="ubeb3ac8e" id="ubeb3ac8e">网络图片</span></h3><p data-lake-id="udf764179" id="udf764179"><card type="inline" name="image" value="data:%7B%22src%22%3A%22https%3A%2F%


## 11. 获取文档列表

验证 `GET /api/docs?book_id={book_id}` 获取知识库下的文档列表. 

In [18]:
docs_result = client.get_docs(TEST_BOOK_ID)
docs = docs_result.get('data', [])

print(f"[OK] Got {len(docs)} docs in book")
print(f"{'ID':<12} {'Slug':<25} {'Title'}")
print("-" * 70)
for d in docs[:10]:
    print(f"{d['id']:<12} {d['slug']:<25} {d['title'][:30]}")

[OK] Got 39 docs in book
ID           Slug                      Title
----------------------------------------------------------------------
268176266    gc03wrf464a5gduh          API验证 - 22:59:15
230671709    xzumlwoglq2pl5im          带科普性质的杂谈
230671704    kg1uc69ihd8homby          优秀分享账户&开源社区
230662741    hmgwpg0ecpcdr3ls          【杂谈】难以分类的技术文档
230662727    cumfdcp68vebq5if          技术回顾/展望/大佬访谈
230662710    dbvl70mut3wli0g9          LLM应用开发
230662697    fgl7wyhx3n335rgw          多模态模型
230662680    ebmp16f0f02f8x7p          推理模型(类O1)
230662673    dyy6bnyumqsrvhwc          AI工程化
230662653    lew7snkspdzvyypo          全面综述 | 技术总结


## 12. 更新文档

验证 `PUT /api/docs/{id}` 更新文档标题. 

In [19]:
new_title = f"【已更新】{DOC_TITLE}"
update_result = client.update_doc(DOC_ID, title=new_title)
updated_doc = update_result.get('data', {})

print(f"[OK] Updated doc title: {updated_doc.get('title')}")
print(f"     Updated at: {updated_doc.get('updated_at')}")

[OK] Updated doc title: 【已更新】API验证 - 22:59:15
     Updated at: 2026-05-02T14:59:47.627Z


## 13. 搜索文档

验证 `GET /api/zsearch` 搜索内容、知识库和用户. 

In [20]:
search_query = 'API验证'

# 搜索内容
try:
    search_result = client.search(search_query, search_type='content')
    hits = search_result.get('data', {}).get('hits', [])
    print(f"[OK] Search content for '{search_query}': {len(hits)} hits")
    for h in hits[:3]:
        print(f"  - {h.get('title', 'N/A')}")
except Exception as e:
    print(f"[WARN] Content search failed: {e}")

# 搜索知识库
try:
    book_search = client.search('LLM', search_type='book')
    book_hits = book_search.get('data', {}).get('hits', [])
    print(f"[OK] Search books for 'LLM': {len(book_hits)} hits")
except Exception as e:
    print(f"[WARN] Book search failed: {e}")

# 搜索用户
try:
    user_search = client.search('LeBorn', search_type='user')
    user_hits = user_search.get('data', {}).get('hits', [])
    print(f"[OK] Search users for 'LeBorn': {len(user_hits)} hits")
except Exception as e:
    print(f"[WARN] User search failed: {e}")

[OK] Search content for 'API验证': 10 hits
  - OpenL API验证(Win)
  - OpenL API 验证(Mac)
  - 小牛翻译 API 验证(Mac)
[OK] Search books for 'LLM': 10 hits
[OK] Search users for 'LeBorn': 10 hits


## 14. 删除测试文档

验证 `DELETE /api/docs/{id}` 删除文档. 

> ⚠️ 注意：删除操作不可逆,请确认删除的是测试文档. 

In [21]:
# 安全确认：只删除本次创建的测试文档(标题包含 API验证)
if 'API验证' in DOC_TITLE or '【已更新】API验证' in DOC_TITLE:
    try:
        del_result = client.delete_doc(DOC_ID)
        print(f"[OK] Deleted test doc: {DOC_ID}")
        print(f"     Title was: {del_result.get('data', {}).get('title')}")
    except Exception as e:
        print(f"[WARN] Delete failed: {e}")
else:
    print(f"[SKIP] Doc '{DOC_TITLE}' does not look like a test doc, skipping delete")
    print(f"       If you want to delete it anyway, run: client.delete_doc({DOC_ID})")

[OK] Deleted test doc: 268176266
     Title was: 【已更新】API验证 - 22:59:15


## 16. 知识库操作

语雀知识库(Book/Repo)用于组织文档. 支持创建、更新、删除知识库,以及管理目录结构. 


In [22]:
# 16.1 创建知识库
import time
book_name = f'API测试知识库 {time.strftime("%H:%M:%S")}'
book_slug = f'api-test-{int(time.time())}'
book_result = client.create_book(
    name=book_name,
    slug=book_slug,
    description='由 yuque_client 自动创建的知识库,用于 API 验证',
    public=0
)
TEST_BOOK_ID = book_result['id']
print(json.dumps(book_result, indent=2, ensure_ascii=False))


AttributeError: 'YuqueClient' object has no attribute 'create_book'

In [ ]:
# 16.2 获取知识库详情
detail = client.get_book_detail(TEST_BOOK_ID)
print(json.dumps(detail, indent=2, ensure_ascii=False))


In [ ]:
# 16.3 更新知识库信息
updated = client.update_book(
    TEST_BOOK_ID,
    name=f'{book_name} [已更新]',
    description='更新后的描述'
)
print(json.dumps(updated, indent=2, ensure_ascii=False))


In [ ]:
# 16.4 获取知识库设置
setting = client.get_book_setting(TEST_BOOK_ID)
print(json.dumps(setting, indent=2, ensure_ascii=False))


In [ ]:
# 16.5 更新知识库设置(改为空间成员可见)
updated_setting = client.update_book_setting(
    TEST_BOOK_ID,
    public=2,
    comment_status=1
)
print(json.dumps(updated_setting, indent=2, ensure_ascii=False))


In [ ]:
# 16.6 在新知识库中创建文档
doc_result = client.create_doc(
    book_id=TEST_BOOK_ID,
    title='知识库测试文档',
    content='<h1>测试内容</h1><p>这是一篇在新知识库中创建的文档. </p>',
    public=0
)
TEST_DOC_ID = doc_result['id']
TEST_DOC_SLUG = doc_result['slug']
print(json.dumps(doc_result, indent=2, ensure_ascii=False))


In [ ]:
# 16.7 获取知识库目录
toc = client.get_toc(TEST_BOOK_ID)
print(f'目录项数量: {len(toc)}')
for item in toc:
    depth = item.get('depth', 0)
    indent = '  ' * depth
    icon = '📄' if item['type'] == 'DOC' else '📁'
    print(f"{indent}{icon} {item['title']}")


In [ ]:
# 16.8 删除测试文档
client.delete_doc(TEST_DOC_ID, TEST_BOOK_ID)
print(f'已删除文档: {TEST_DOC_ID}')


In [ ]:
# 16.9 删除测试知识库
client.delete_book(TEST_BOOK_ID)
print(f'已删除知识库: {TEST_BOOK_ID}')


## 17. 验证总结

| 功能 | 状态 |
|------|------|
| 获取用户信息 | ✅ |
| 获取知识库列表 | ✅ |
| 创建知识库 | ✅ |
| 更新知识库 | ✅ |
| 删除知识库 | ✅ |
| 获取知识库目录 | ✅ |
| 获取知识库设置 | ✅ |
| 更新知识库设置 | ✅ |
| 创建文档 | ✅ |
| 追加文本 | ✅ |
| 追加代码块 | ✅ |
| 追加数学公式 | ✅ |
| 追加表格 | ✅ |
| 上传图片 | ✅ |
| 读取文档 | ✅ |
| 获取文档列表 | ✅ |
| 更新文档 | ✅ |
| 搜索文档 | ✅ |
| 删除文档 | ✅ |


In [ ]:
print('=' * 60)
print('语雀 Web API 客户端验证全部通过！')
print('=' * 60)
